#1. Install Required Packages

In [ ]:
!pip install -q -U \
    langchain \
    langchain-core \
    langchain-chroma \
    langchain-huggingface \
    sentence-transformers \
    chromadb \
    pandas \
    openpyxl

#2. Import Libraries

In [ ]:
# ==========================================================
# Imports
# ==========================================================

import os
import re
import gc
import torch
import pandas as pd

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

print("Imports successful.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#Define Project Paths

In [ ]:
# ==========================================================
# Project Paths
# ==========================================================

PROJECT_DIR = "/content/drive/MyDrive/RAG_ICH"

DATA_DIR = os.path.join(PROJECT_DIR, "data")
CHUNK_DIR = os.path.join(DATA_DIR, "chunks")
EVALUATION_DIR = os.path.join(DATA_DIR, "evaluation")

VECTOR_DIR = os.path.join(PROJECT_DIR, "vector_db")
RESULT_DIR = os.path.join(PROJECT_DIR, "results", "embedding")

# Create folders if they don't exist
os.makedirs(VECTOR_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

print("CHUNK_DIR:")
print(CHUNK_DIR)

print("\nEVALUATION_DIR:")
print(EVALUATION_DIR)

print("\nVECTOR_DIR:")
print(VECTOR_DIR)

print("\nRESULT_DIR:")
print(RESULT_DIR)

#Fix Chunking Strategy to Recursive Chunking

In [ ]:
# ==========================================================
# Fixed Experimental Settings
# ==========================================================

CHUNK_FILE = "recursive_chunks.csv"

CHUNK_PATH = os.path.join(
    CHUNK_DIR,
    CHUNK_FILE
)

EVAL_FILE = os.path.join(
    EVALUATION_DIR,
    "evaluation_questions.xlsx"
)

TOP_K = 5

print("Chunk file:")
print(CHUNK_PATH)

print("\nEvaluation file:")
print(EVAL_FILE)

print(f"\nTop-k: {TOP_K}")

In [ ]:
print("Chunk file exists:", os.path.exists(CHUNK_PATH))
print("Evaluation file exists:", os.path.exists(EVAL_FILE))

In [ ]:
# ==========================================================
# Check GPU
# ==========================================================

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    DEVICE = "cuda"
else:
    print("Using CPU")
    DEVICE = "cpu"

In [ ]:
# ==========================================================
# Load Recursive Chunks
# ==========================================================

chunk_df = pd.read_csv(CHUNK_PATH)

print(f"Number of chunks: {len(chunk_df)}")

display(chunk_df.head())

#Convert Chunk into LangChain Documents

In [ ]:
# ==========================================================
# Create LangChain Documents
# ==========================================================

documents = []

for _, row in chunk_df.iterrows():

    doc = Document(
        page_content=row["text"],
        metadata={
            "chunk_id": int(row["chunk_id"]),
            "chunking_method": "recursive"
        }
    )

    documents.append(doc)

print(f"Created {len(documents)} documents.")

#Load Evalution Questions

In [ ]:
# ==========================================================
# Load Evaluation Questions
# ==========================================================

evaluation_df = pd.read_excel(EVAL_FILE)

print(f"Number of evaluation questions: {len(evaluation_df)}")

print("\nColumns:")
print(evaluation_df.columns.tolist())

display(evaluation_df.head())

In [ ]:
# ==========================================================
# Embedding Models to Compare
# ==========================================================

EMBEDDING_MODELS = {

    "all_mpnet_base_v2":
        "sentence-transformers/all-mpnet-base-v2",

    "BioLORD_2023_C":
        "FremyCompany/BioLORD-2023-C"

}

EMBEDDING_MODELS

In [ ]:
# ==========================================================
# Function: Run Embedding Model Experiment
# ==========================================================

def run_embedding_experiment(
    model_label,
    model_name,
    documents,
    evaluation_df,
    top_k=5
):

    print("=" * 70)
    print(f"RUNNING MODEL: {model_label}")
    print(f"MODEL NAME: {model_name}")
    print("=" * 70)

    # ------------------------------------------------------
    # Load embedding model
    # ------------------------------------------------------

    embedding_model = HuggingFaceEmbeddings(

        model_name=model_name,

        model_kwargs={
            "device": DEVICE
        },

        encode_kwargs={
            "normalize_embeddings": True
        }

    )

    print("\nEmbedding model loaded successfully.")

    # ------------------------------------------------------
    # Define Chroma collection
    # ------------------------------------------------------

    collection_name = f"recursive_{model_label}"

    persist_directory = os.path.join(
        VECTOR_DIR,
        "embedding_evaluation",
        model_label
    )

    os.makedirs(persist_directory, exist_ok=True)

    print("\nCreating ChromaDB...")
    print("Collection:", collection_name)

    # ------------------------------------------------------
    # Create vector database
    # ------------------------------------------------------

    vector_db = Chroma.from_documents(

        documents=documents,

        embedding=embedding_model,

        collection_name=collection_name,

        persist_directory=persist_directory

    )

    print("ChromaDB created successfully.")

    # ------------------------------------------------------
    # Create retriever
    # ------------------------------------------------------

    retriever = vector_db.as_retriever(

        search_type="similarity",

        search_kwargs={
            "k": top_k
        }

    )

    print(f"Retriever created with k={top_k}")

    # ------------------------------------------------------
    # Retrieve Top-k chunks
    # ------------------------------------------------------

    retrieval_results = []

    print("\nRetrieving chunks...")

    for _, row in evaluation_df.iterrows():

        question_id = row["question_id"]

        question = row["clinical_question"]

        docs = retriever.invoke(question)

        for rank, doc in enumerate(docs, start=1):

            retrieval_results.append({

                "question_id": question_id,

                "domain": row["domain"],

                "subdomain": row["subdomain"],

                "clinical_question": question,

                "reference_answer_draft":
                    row["reference_answer_draft"],

                "gold_evidence_location":
                    row["gold_evidence_location"],

                "required_gold_evidence_for_retrieval":
                    row["required_gold_evidence_for_retrieval"],

                "rank": rank,

                "chunk_id":
                    doc.metadata["chunk_id"],

                "retrieved_text":
                    doc.page_content

            })

    # ------------------------------------------------------
    # Create DataFrame
    # ------------------------------------------------------

    retrieval_df = pd.DataFrame(
        retrieval_results
    )

    # ------------------------------------------------------
    # Save results
    # ------------------------------------------------------

    output_file = os.path.join(

        RESULT_DIR,

        f"{model_label}_retrieval_results.csv"

    )

    retrieval_df.to_csv(
        output_file,
        index=False
    )

    print("\n" + "=" * 70)
    print("EXPERIMENT COMPLETED")
    print("=" * 70)

    print(f"Model: {model_label}")

    print(
        f"Total retrieval rows: "
        f"{len(retrieval_df)}"
    )

    print(f"Results saved to:")

    print(output_file)

    # ------------------------------------------------------
    # Clean GPU memory
    # ------------------------------------------------------

    del vector_db
    del embedding_model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

    return retrieval_df

#Experiment 1: Run Baseline Model

In [ ]:
# ==========================================================
# Experiment 1
# all-mpnet-base-v2
# ==========================================================

mpnet_results = run_embedding_experiment(

    model_label="all_mpnet_base_v2",

    model_name=
        "sentence-transformers/all-mpnet-base-v2",

    documents=documents,

    evaluation_df=evaluation_df,

    top_k=TOP_K

)

In [ ]:
print("Number of rows:", len(mpnet_results))

display(mpnet_results.head(10))

In [ ]:
# ==========================================================
# Experiment 2
# BioLORD-2023-C
# ==========================================================

biolord_results = run_embedding_experiment(

    model_label="BioLORD_2023_C",

    model_name=
        "FremyCompany/BioLORD-2023-C",

    documents=documents,

    evaluation_df=evaluation_df,

    top_k=TOP_K

)

In [ ]:
# ==========================================================
# Verify Output Files
# ==========================================================

print("all-mpnet-base-v2:")
print(
    os.path.exists(
        os.path.join(
            RESULT_DIR,
            "all_mpnet_base_v2_retrieval_results.csv"
        )
    )
)

print("\nBioLORD-2023-C:")
print(
    os.path.exists(
        os.path.join(
            RESULT_DIR,
            "BioLORD_2023_C_retrieval_results.csv"
        )
    )
)